<div style="background-color: red; padding: 10px; border-radius: 10px; text-align: center;">
  <span style="color: white; font-size: 25px; font-weight: bold;">
      Intel Image Classification using CNN
    
   </span>
</div>


<span style="color: Blue; font-size: 18px; font-weight: bold;">
      By : Ahmed Almohamdy
    
   </span>


________

 <span style="color: red; font-size: 18px; font-weight: bold;">
    
   Building Model using CNN by tensorflow and Keras to Classify 14k+ Image to 6 category
    
   </span>


**Data link** : https://www.kaggle.com/puneet6060/intel-image-classification



<div style="background-color: green; padding: 10px; border-radius: 10px; text-align: center;">
  <span style="color: white; font-size: 15px; font-weight: bold;">
      Importing
    
   </span>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set(style="whitegrid")
import os
import glob as gb
import cv2
import tensorflow as tf
import keras
from sklearn.metrics import confusion_matrix



# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [ ]:
trainpath = '/kaggle/input/intel-image-classification/seg_train/'
testpath = '/kaggle/input/intel-image-classification/seg_test/'
predpath = '/kaggle/input/intel-image-classification/seg_pred/'

# **Open Folders**

In [ ]:
for folder in  os.listdir(trainpath + 'seg_train') : 
    files = gb.glob(pathname= str( trainpath +'seg_train//' + folder + '/*.jpg'))
    print(f'For training data , found {len(files)} in folder {folder}')

In [ ]:
for folder in  os.listdir(testpath +'seg_test') : 
    files = gb.glob(pathname= str( testpath +'seg_test//' + folder + '/*.jpg'))
    print(f'For testing data , found {len(files)} in folder {folder}')

_____
now for prediction folder

In [ ]:
files = gb.glob(pathname= str(predpath +'seg_pred/*.jpg'))
print(f'For Prediction data , found {len(files)}')

# Checking Images

**now we need to heck the images sizes , to know how they looks like**

**since we have 6 categories , we first need to create a dictionary with their names & indices , also create a function to get the code back**

In [ ]:
code = {'buildings':0 ,'forest':1,'glacier':2,'mountain':3,'sea':4,'street':5} # tell me this in data description

def getcode(n) : 
    for x , y in code.items() : 
        if n == y : 
            return x    

In [ ]:
size = []
for folder in  os.listdir(trainpath +'seg_train') : 
    files = gb.glob(pathname= str( trainpath +'seg_train//' + folder + '/*.jpg'))
    for file in files: 
        image = plt.imread(file)
        size.append(image.shape)
pd.Series(size).value_counts()

<span style="color: red; font-size: 15px; font-weight: bold;">
      most is (150 * 150 * 3 )
    
   </span>


In [ ]:
size = []
for folder in  os.listdir(testpath +'seg_test') : 
    files = gb.glob(pathname= str( testpath +'seg_test//' + folder + '/*.jpg'))
    for file in files: 
        image = plt.imread(file)
        size.append(image.shape)
pd.Series(size).value_counts()

In [ ]:
size = []
files = gb.glob(pathname= str(predpath +'seg_pred/*.jpg'))
for file in files: 
    image = plt.imread(file)
    size.append(image.shape)
pd.Series(size).value_counts()

# Reading Images

**now it's time to read all images & convert it into arrays**
**let's use now size = 100 , so it will be suitable amount to contain accuracy without losing so much time in training**

In [ ]:
# new size (100,100,3)
s = 100

# **Now to read all pictues in six categories in training folder,using OpenCV to resize it**

In [ ]:
# image (hight , width , depth(RGB) ) --- > if i have (100 , 100 , 3) then (0:50 ,: , :) this will the top half of image
#                                                                          (: ,0:50 ,:) this will the left half of image

X_train = []
y_train = []
for folder in  os.listdir(trainpath +'seg_train') : 
    files = gb.glob(pathname= str( trainpath +'seg_train//' + folder + '/*.jpg'))
    for file in files: 
        image = cv2.imread(file) # convert image to array
        image_array = cv2.resize(image , (s,s)) # resize first two dimention (150) dimintion 3 never change
        X_train.append(list(image_array))
        y_train.append(code[folder])

In [ ]:
print(len(X_train))
print(len(y_train))

<div style="background-color: green; padding: 10px; border-radius: 10px; text-align: center;">
  <span style="color: white; font-size: 15px; font-weight: bold;">
      Preparing Test and Train Data and Visualize It
    
   </span>
</div>

In [ ]:
fig, ax = plt.subplots(nrows = 5, ncols = 5, figsize=(20, 20))
ax = ax.flatten()
i=0
for n in list(np.random.randint(0,len(X_train),25)):
    ax[i].imshow(X_train[n])
    ax[i].axis('off')
    ax[i].grid(False)
    ax[i].set_title(getcode(y_train[n]))
    i+=1


plt.show()

In [ ]:
X_test = []
y_test = []
for folder in  os.listdir(testpath +'seg_test') : 
    files = gb.glob(pathname= str(testpath + 'seg_test//' + folder + '/*.jpg'))
    for file in files: 
        image = cv2.imread(file)
        image_array = cv2.resize(image , (s,s))
        X_test.append(list(image_array))
        y_test.append(code[folder])
        

In [ ]:
print(len(X_test))
print(len(y_test))

In [ ]:
fig, ax = plt.subplots(nrows = 5, ncols = 5, figsize=(20, 20))
ax = ax.flatten()
i=0
for n in list(np.random.randint(0,len(X_test),25)):
    ax[i].imshow(X_test[n])
    ax[i].axis('off')
    ax[i].grid(False)
    ax[i].set_title(getcode(y_test[n]))
    i+=1


plt.show()

In [ ]:
X_pred = []
files = gb.glob(pathname= str(predpath + 'seg_pred/*.jpg'))
for file in files: 
    image = cv2.imread(file)
    image_array = cv2.resize(image , (s,s))
    X_pred.append(list(image_array))       

In [ ]:
print(len(X_pred))

In [ ]:
fig, ax = plt.subplots(nrows = 5, ncols = 5, figsize=(20, 20))
ax = ax.flatten()
i=0
for n in list(np.random.randint(0,len(X_pred),25)):
    ax[i].imshow(X_pred[n])
    ax[i].axis('off')
    ax[i].grid(False)
    i+=1


plt.show()

<div style="background-color: red; padding: 10px; border-radius: 10px; text-align: center;">
  <span style="color: white; font-size: 15px; font-weight: bold;">
      Building Model
    
   </span>
</div>



**Convert lists to numpy arrays**

In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)
X_pred_array = np.array(X_pred)
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f'X_train shape  is {X_train.shape}')
print(f'X_test shape  is {X_test.shape}')
print(f'X_pred shape  is {X_pred_array.shape}')
print(f'y_train shape  is {y_train.shape}')
print(f'y_test shape  is {y_test.shape}')

# **Building the CNN model by Keras , using Conv2D layers , MaxPooling ,Batch Normalization & Denses**

In [ ]:
KerasModel = keras.models.Sequential([
        keras.layers.Input((s,s,3)),
    
        keras.layers.Conv2D(128,kernel_size=(3,3),activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(pool_size=(2,2)),
    
        keras.layers.Conv2D(64,kernel_size=(3,3),activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(pool_size=(2,2)),
    
        keras.layers.Conv2D(32,kernel_size=(3,3),activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPool2D(pool_size=(2,2)),
    
      
        keras.layers.Flatten() ,   
    
       
        keras.layers.Dense(128,activation='relu') ,
        keras.layers.Dropout(rate=0.4) ,       
    
        keras.layers.Dense(6,activation='softmax') ,    
        ])

# **Compiling the model , using adam optimizer , & sparse categorical crossentropy loss**

In [ ]:
KerasModel.compile(optimizer ='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

# **How the model looks like ?**

In [ ]:
print(KerasModel.summary())

# **training the model , lets use 25 epochs now**

In [ ]:
epochs = 25
ThisModel = KerasModel.fit(X_train, y_train, epochs=epochs,batch_size=128,verbose=1)

# **loss & accuracy**


In [ ]:
ModelLoss, ModelAccuracy = KerasModel.evaluate(X_test, y_test)

print(f'Test Loss is {ModelLoss}')
print(f'Test Accuracy is {ModelAccuracy}')

In [ ]:
y_pred = KerasModel.predict(X_test)

print(f'Prediction Shape is {y_pred.shape}')

In [ ]:
y_result = KerasModel.predict(X_pred_array)

print('Prediction Shape is {}'.format(y_result.shape))

In [ ]:
fig, ax = plt.subplots(nrows = 6, ncols = 6, figsize=(20, 20))
ax = ax.flatten()
i=0
for n in list(np.random.randint(0,len(X_pred),36)):
    ax[i].imshow(X_pred[n])
    ax[i].axis('off')
    ax[i].grid(False)
    ax[i].set_title(getcode(np.argmax(y_result[n])))
    i+=1


plt.show()

<div style="background-color: green; padding: 10px; border-radius: 10px; text-align: center;">
  <span style="color: white; font-size: 15px; font-weight: bold;">
      plot confusion matrix To can evaluate  Model
    
   </span>
</div>

In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, class_names):
    
    cm = confusion_matrix(true_labels, pred_labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    
    plt.title("Confusion Matrix", fontsize=16)
    plt.xlabel("Predicted Labels", fontsize=12)
    plt.ylabel("True Labels", fontsize=12)
    plt.tight_layout()
    plt.show()

class_names = ['buildings','forest','glacier','mountain','sea','street' ]   
predictions = KerasModel.predict(X_test) 
pred_labels = np.argmax(predictions, axis=1)  
plot_confusion_matrix(y_test, pred_labels, class_names)